<div class="alert alert-block alert-info">
<b>Welcome to SynEdu!</b> This talktorial is part of <b>SynEdu</b>, a lightweight teaching series built around the <b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible cheminformatics.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility first</b>: Keep runtime short, prefer small datasets, and pin dependencies (e.g., via <code>env/environment.yml</code>). Save version info alongside exported figures.
</div>

<div class="alert alert-block alert-success">
<b>By the end of this notebook</b>, you will extract a <b>minimal rule</b> (L/K/R) from one atom-mapped reaction using the bond-delta reaction center.
</div>

# S04 · Extract a minimal L/K/R rule from one mapped reaction

**Data:** `data/reactions_mapped.csv`


## Authors and contributions

- Tieu-Long Phan, Peter Stadler group, Professur für Bioinformatik, Institut für Informatik, Universität Leipzig
- (Add contributors here)


<div class="alert alert-block alert-info">
<b>Cross-referencing</b>: When referring to another SynEdu notebook, use <b>Talktorial SXX</b> (e.g., <b>Talktorial S03</b>).
</div>


## Roadmap
- Concepts: center atoms → L/K/R
- Hands-on: extract one rule and save to JSON


# Theory

We build a rule from a mapped reaction by:
1) Find changed mapped bonds (created/deleted)
2) Collect involved mapped atoms (reaction center atoms)
3) Build subgraphs in reactant (L) and product (R) induced by those atoms
4) Let K be the intersection of L and R (shared atoms)

This is the smallest teaching extractor. Context (radius) comes later.


# Practical


In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw

# Optional: Syn ecosystem (kept optional for Paper 1)
try:
    import synkit  # type: ignore
    HAS_SYNKit = True
except Exception:
    HAS_SYNKit = False

OUT = Path("talktorials/out")
OUT.mkdir(parents=True, exist_ok=True)

import rdkit
import networkx as nx_mod
print("RDKit:", rdkit.__version__)
print("NetworkX:", nx_mod.__version__)
print("SynKit available:", HAS_SYNKit)


In [ ]:
import json

df = pd.read_csv("data/reactions_mapped.csv")
row = df.iloc[0]
react, prod = row.am_rxn_smiles.split(">>")

mR = Chem.MolFromSmiles(react)
mP = Chem.MolFromSmiles(prod)


In [ ]:
def mol_to_mapped_graph(m: Chem.Mol) -> nx.Graph:
    G = nx.Graph()
    for a in m.GetAtoms():
        amap = a.GetAtomMapNum()
        if amap:
            G.add_node(amap, symbol=a.GetSymbol(), charge=int(a.GetFormalCharge()), aromatic=bool(a.GetIsAromatic()))
    for b in m.GetBonds():
        ai=b.GetBeginAtom().GetAtomMapNum(); aj=b.GetEndAtom().GetAtomMapNum()
        if ai and aj and ai in G.nodes and aj in G.nodes:
            G.add_edge(min(ai,aj), max(ai,aj), order=int(b.GetBondTypeAsDouble()), aromatic=bool(b.GetIsAromatic()))
    return G

def changed_pairs(mR: Chem.Mol, mP: Chem.Mol) -> tuple[set[tuple[int,int]], set[tuple[int,int]]]:
    def pairs(m):
        out=set()
        for b in m.GetBonds():
            ai=b.GetBeginAtom().GetAtomMapNum(); aj=b.GetEndAtom().GetAtomMapNum()
            if ai and aj:
                out.add((min(ai,aj), max(ai,aj)))
        return out
    pR=pairs(mR); pP=pairs(mP)
    return (pP-pR), (pR-pP)

created, deleted = changed_pairs(mR, mP)
center_atoms = set([x for ab in (created | deleted) for x in ab])

GR = mol_to_mapped_graph(mR)
GP = mol_to_mapped_graph(mP)

L = GR.subgraph(center_atoms).copy()
R = GP.subgraph(center_atoms).copy()
K_nodes = set(L.nodes()).intersection(R.nodes())
K = L.subgraph(K_nodes).copy()

rule = {
    "rxn_id": row.rxn_id,
    "label": row.label,
    "created": sorted(list(created)),
    "deleted": sorted(list(deleted)),
    "L": nx.node_link_data(L),
    "K": nx.node_link_data(K),
    "R": nx.node_link_data(R),
}
rule


In [ ]:
# Save rule for later notebooks
out_path = OUT / "S04_rule.json"
out_path.write_text(json.dumps(rule, indent=2), encoding="utf-8")
print("Wrote", out_path)


# Discussion
- This minimal rule can be too specific or too weak depending on the reaction.
- Adding context (radius=1/2) often improves specificity, but increases rule size.


# Quiz
1. What would go wrong if we define K as empty for every rule?
2. How would you include bond order changes in the center?
3. Add an option `radius=1` to include 1-hop neighbors around the center atoms.


# References and further reading

*Suggested citation style:*  
* Keyword: <i>Source</i> (year) (link)

- RDKit documentation: <i>RDKit</i> (ongoing) — https://www.rdkit.org/docs/
- RDKit Book: <i>The RDKit Book</i> (ongoing) — https://www.rdkit.org/docs/Book.html
- NetworkX documentation: <i>NetworkX</i> (ongoing) — https://networkx.org/documentation/stable/
- Graphviz DOT language: <i>Graphviz</i> (ongoing) — https://graphviz.org/documentation/
